In [2]:
# Шаг 1: Импорты и базовые переменные/константы
from copy import deepcopy

import numpy as np
from gymnasium import spaces
from numba import njit

from gym_art.quadrotor_multi.inertia import QuadLink, QuadLinkSimplified
from gym_art.quadrotor_multi.numba_utils import OUNoiseNumba, angvel2thrust_numba, numba_cross
from gym_art.quadrotor_multi.quad_utils import (
    OUNoise, rand_uniform_rot3d, cross_vec_mx4, cross_mx4, npa, cross,
    randyaw, to_xyhat, normalize
)

GRAV = 9.81   # Постоянная гравитации (м/с^2)
EPS = 1e-6    # Маленькая величина, чтобы избежать деления на ноль и прочих неточностей


In [ ]:
# Примерно то, что раньше было в __init__:
# здесь self.* стали просто глобальными переменными (или вам нужно будет переписать под свой сценарий)

dynamics_steps_num = 1
dim_mode = "3D"
gravity = GRAV
dynamics_simplification = False
use_numba = False
dt = 1/200

prop_ccw = np.array([-1., 1., -1., 1.])
omega_max = 40.0
vxyz_max = 3.0
acc_max = 3.0 * GRAV
since_last_svd = 0
since_last_svd_limit = 0.5
eye = np.eye(3)
thrust_noise = None  # Позже инициализируется в init_thrust_noise()

room_box = np.array([[0., 0., 0.], [10., 10., 10.]])
on_floor = False
floor_threshold = 0.05
mu = 0.6
crashed_wall = False
crashed_ceiling = False
crashed_floor = False

control_mx = None
if dim_mode == '1D':
    control_mx = np.ones([4, 1])
elif dim_mode == '2D':
    control_mx = np.array([[1., 0.], [1., 0.], [0., 1.], [0., 1.]])
elif dim_mode == '3D':
    control_mx = np.eye(4)
else:
    raise ValueError('QuadEnv: Unknown dimensionality mode %s' % dim_mode)

print("Переменные инициализированы.")
